In [1]:
# ------------------------------------------------------------
# 1. Install Dependencies
# ------------------------------------------------------------

!pip install -q nemo_toolkit['asr']
!pip install -q pandas==2.2.2
!pip install -q numpy==1.26.4
!pip install -q numba==0.60.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.1/443.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.3 MB/s eta 0:0

In [2]:
!pip install -q jiwer
!pip install -q hazm
!pip install -q soundfile

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.7 MB/s eta 0:00:00


In [1]:
import os
import re
import time
import pandas as pd
import numpy as np
import soundfile as sf
import torch
import hazm
from tqdm import tqdm
from jiwer import wer, cer
import nemo.collections.asr as nemo_asr

[NeMo W 2026-06-07 09:12:07 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


In [2]:
normalizer = hazm.Normalizer()
PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize(text):
    text = str(text)
    text = normalizer.normalize(text)
    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)
    text = re.sub(r"[^\w\s]", " ", text) # Fixed regex bracket error from original template
    text = text.replace("می ", "می")
    text = text.replace("نمی ", "نمی")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [3]:
TTS_CSV = "/content/drive/MyDrive/tts_benchmark/PIPER_FAS/tts_dataset.csv"
df = pd.read_csv(TTS_CSV)
print("Samples to process:", len(df))

MODEL_NAME = "nvidia/stt_fa_fastconformer_hybrid_large"
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)
asr_model = asr_model.to(device)
print("Using target system device:", device)

Samples to process: 1052
[NeMo I 2026-06-07 09:12:17 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-06-07 09:12:18 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: dummy
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 10
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_lhotse: true
    lhotse:
      shar_path: /data_artifacts/data/shar/train
      batch_duration: 1200
      quadratic_duration: 15
      num_buckets: 10
      num_cuts_for_bins_estimate: 10000
      buffer_size: 10000
      shuffle_buffer_size: 10000
    
[NeMo W 2026-06-07 09:12:18 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a vali

[NeMo I 2026-06-07 09:12:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-07 09:12:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-07 09:12:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-06-07 09:12:20 save_restore_connector:285] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_fa_fastconformer_hybrid_large/snapshots/249cf5bf70dda7220a60ddeeecff2f6aad8e1784/stt_fa_fastconformer_hybrid_large.nemo.
Using target system device: cuda


In [4]:
df_test = df.head(10).copy()

In [4]:
def clean_asr_output(text):
    if text is None:
        return ""

    s = str(text)

    # Extract only the real transcription between "text" and "dec_out"
    match = re.search(r"text\s+(.*?)\s+dec_out", s, flags=re.DOTALL)

    if match:
        return match.group(1).strip()

    return s

def extract_asr_text(output):
    if hasattr(output, "text"):
        return output.text
    elif isinstance(output, dict):
        return output.get("text", str(output))
    else:
        return str(output)

In [6]:
import time
import numpy as np
import soundfile as sf
import pandas as pd
from tqdm import tqdm

# -----------------------------
# STORAGE
# -----------------------------
references = []
predictions = []
audio_paths = []
sample_cers = []

runtimes = []
rtf_list = []

total_audio_duration = 0.0

start_time = time.time()

print("Starting FULL ASR Evaluation...")

# -----------------------------
# LOOP
# -----------------------------
for _, row in tqdm(df.iterrows(), total=len(df)):

    audio_path = row["audio_path"]

    try:
        # -------------------------
        # audio duration
        # -------------------------
        info = sf.info(audio_path)
        duration = info.frames / info.samplerate
        total_audio_duration += duration

        # -------------------------
        # ASR inference (timed)
        # -------------------------
        t0 = time.time()

        output = asr_model.transcribe([audio_path], batch_size=1)[0]
        pred = extract_asr_text(output)

        t1 = time.time()
        runtime = t1 - t0

        # -------------------------
        # normalization
        # -------------------------
        pred = normalize(pred)
        ref = normalize(row["text"])

        # -------------------------
        # store results
        # -------------------------
        predictions.append(pred)
        references.append(ref)
        audio_paths.append(audio_path)

        runtimes.append(runtime)
        rtf_list.append(runtime / duration)

        # -------------------------
        # CER
        # -------------------------
        sample_cers.append(cer(ref, pred))

    except Exception as e:
        print("FAILED:", audio_path)
        print(e)

# -----------------------------
# FINAL STATS
# -----------------------------
end_time = time.time()

asr_time = end_time - start_time

runtimes = np.array(runtimes)
rtf_list = np.array(rtf_list)

asr_rtf = asr_time / total_audio_duration
avg_latency = runtimes.mean()
std_latency = runtimes.std()

mean_rtf = rtf_list.mean()
std_rtf = rtf_list.std()

wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

speedup_vs_realtime = 1 / asr_rtf

# -----------------------------
# PRINT SUMMARY
# -----------------------------
print("\n" + "="*50)
print("FULL ASR EVALUATION SUMMARY")
print("="*50)
print(f"Samples: {len(predictions)}")
print(f"WER: {wer_score:.4f}")
print(f"CER: {cer_score:.4f}")
print(f"ASR RTF (aggregate): {asr_rtf:.4f}")
print(f"Mean RTF: {mean_rtf:.4f}")
print(f"Std RTF: {std_rtf:.4f}")
print(f"Avg latency: {avg_latency:.4f} sec")
print(f"Std latency: {std_latency:.4f}")
print(f"Speedup vs realtime: {speedup_vs_realtime:.2f}x")
print("="*50)

# -----------------------------
# SAVE RESULTS (PER SAMPLE)
# -----------------------------
results_df = pd.DataFrame({
    "audio_path": audio_paths,
    "reference": references,
    "prediction": predictions,
    "sample_cer": sample_cers
})

SAVE_PATH = "/content/drive/MyDrive/tts_asr_results_full.csv"

results_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("Saved per-sample results to:")
print(SAVE_PATH)

# -----------------------------
# SAVE SUMMARY
# -----------------------------
summary_df = pd.DataFrame([{
    "model": "Piper_Amir_Medium",

    "samples": len(predictions),

    "wer": wer_score,
    "cer": cer_score,

    "asr_rtf": asr_rtf,
    "mean_rtf": mean_rtf,
    "std_rtf": std_rtf,

    "avg_latency_sec": avg_latency,
    "std_latency_sec": std_latency,

    "speedup_vs_realtime": speedup_vs_realtime
}])

SUMMARY_PATH = "/content/drive/MyDrive/tts_asr_summary_piper.csv"

summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Saved summary to:")
print(SUMMARY_PATH)

Starting FULL ASR Evaluation...


  0%|          | 0/1052 [00:00<?, ?it/s][NeMo W 2026-06-07 09:14:57 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-07 09:14:58 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 11.46it/s]
  0%|          | 1/1052 [00:00<02:49,  6.19it/s][NeMo W 2026-06-07 09:14:58 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-07 09:14:58 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenizat


FULL ASR EVALUATION SUMMARY
Samples: 1052
WER: 0.1071
CER: 0.0455
ASR RTF (aggregate): 0.0613
Mean RTF: 0.0738
Std RTF: 0.0415
Avg latency: 0.1316 sec
Std latency: 0.0306
Speedup vs realtime: 16.31x
Saved per-sample results to:
/content/drive/MyDrive/tts_asr_results_full.csv
Saved summary to:
/content/drive/MyDrive/tts_asr_summary_piper.csv


In [27]:
end_time = time.time()
asr_time = end_time - start_time

asr_rtf = asr_time / total_audio_duration
avg_latency = asr_time / len(predictions)

wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

In [28]:
print("\n" + "="*50)
print("FULL ASR EVALUATION SUMMARY")
print("="*50)
print(f"Samples: {len(predictions)}")
print(f"Total ASR time: {asr_time:.2f} sec")
print(f"Total audio duration: {total_audio_duration:.2f} sec")
print(f"ASR RTF: {asr_rtf:.4f}")
print(f"Avg latency: {avg_latency:.4f} sec")
print(f"WER: {wer_score:.4f}")
print(f"CER: {cer_score:.4f}")
print("="*50)


FULL ASR EVALUATION SUMMARY
Samples: 1052
Total ASR time: 996.96 sec
Total audio duration: 2376.11 sec
ASR RTF: 0.4196
Avg latency: 0.9477 sec
WER: 0.1071
CER: 0.0455


In [29]:
results_df = pd.DataFrame({
    "audio_path": audio_paths,
    "reference": references,
    "prediction": predictions,
    "sample_cer": sample_cers
})

SAVE_PATH = "/content/drive/MyDrive/tts_asr_results_full.csv"

results_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print(f"\nSaved results to: {SAVE_PATH}")


Saved results to: /content/drive/MyDrive/tts_asr_results_full.csv


In [32]:
summary_df = pd.DataFrame([{
    "model": "PIPER_fa_IR-gyro-medium",   # change per experiment

    "samples": len(predictions),

    "wer": wer_score,
    "cer": cer_score,

    "asr_rtf": asr_rtf,
    "avg_latency_sec": avg_latency,

    "total_asr_time_sec": asr_time,
    "total_audio_duration_sec": total_audio_duration
}])

print(summary_df)

                     model  samples       wer       cer   asr_rtf  \
0  PIPER_fa_IR-gyro-medium     1052  0.107082  0.045468  0.419576   

   avg_latency_sec  total_asr_time_sec  total_audio_duration_sec  
0          0.94768          996.959188               2376.109569  


In [33]:
SUMMARY_PATH = "/content/drive/MyDrive/tts_asr_summary_piper.csv"

summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Saved ASR summary to:")
print(SUMMARY_PATH)

Saved ASR summary to:
/content/drive/MyDrive/tts_asr_summary_piper.csv


In [35]:
runtimes = []
rtf_list = []

In [36]:
t0 = time.time()

output = asr_model.transcribe([audio_path], batch_size=1)[0]
pred = extract_asr_text(output)

t1 = time.time()
runtime = t1 - t0

runtimes.append(runtime)
rtf_list.append(runtime / duration)

[NeMo W 2026-06-07 08:56:25 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-07 08:56:25 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00, 21.73it/s]


In [37]:
import numpy as np

runtimes = np.array(runtimes)
rtf_list = np.array(rtf_list)

aggregate_rtf = asr_time / total_audio_duration
mean_rtf = rtf_list.mean()
std_rtf = rtf_list.std()

avg_latency_sec = runtimes.mean()
std_latency_sec = runtimes.std()

speedup_vs_realtime = 1 / aggregate_rtf

In [38]:
summary_df = pd.DataFrame([{
    "model": "Piper_Amir_Medium",   # change per model

    "samples": len(predictions),

    "aggregate_rtf": aggregate_rtf,
    "mean_rtf": mean_rtf,
    "std_rtf": std_rtf,

    "avg_latency_sec": avg_latency_sec,
    "std_latency_sec": std_latency_sec,

    "speedup_vs_realtime": speedup_vs_realtime,

    "wer": wer_score,
    "cer": cer_score
}])

print(summary_df)

               model  samples  aggregate_rtf  mean_rtf  std_rtf  \
0  Piper_Amir_Medium     1052       0.419576  0.124724      0.0   

   avg_latency_sec  std_latency_sec  speedup_vs_realtime       wer       cer  
0         0.083987              0.0             2.383357  0.107082  0.045468  


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/tts_asr_summary_full.csv"

summary_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("Saved:", SAVE_PATH)

In [39]:
import time
import numpy as np
import soundfile as sf
import pandas as pd
from tqdm import tqdm

# -----------------------------
# STORAGE
# -----------------------------
references = []
predictions = []
audio_paths = []
sample_cers = []

runtimes = []
rtf_list = []

total_audio_duration = 0.0

start_time = time.time()

print("Starting FULL ASR Evaluation...")

# -----------------------------
# LOOP
# -----------------------------
for _, row in tqdm(df.iterrows(), total=len(df)):

    audio_path = row["audio_path"]

    try:
        # -------------------------
        # audio duration
        # -------------------------
        info = sf.info(audio_path)
        duration = info.frames / info.samplerate
        total_audio_duration += duration

        # -------------------------
        # ASR inference (timed)
        # -------------------------
        t0 = time.time()

        output = asr_model.transcribe([audio_path], batch_size=1)[0]
        pred = extract_asr_text(output)

        t1 = time.time()
        runtime = t1 - t0

        # -------------------------
        # normalization
        # -------------------------
        pred = normalize(pred)
        ref = normalize(row["text"])

        # -------------------------
        # store results
        # -------------------------
        predictions.append(pred)
        references.append(ref)
        audio_paths.append(audio_path)

        runtimes.append(runtime)
        rtf_list.append(runtime / duration)

        # -------------------------
        # CER
        # -------------------------
        sample_cers.append(cer(ref, pred))

    except Exception as e:
        print("FAILED:", audio_path)
        print(e)

# -----------------------------
# FINAL STATS
# -----------------------------
end_time = time.time()

asr_time = end_time - start_time

runtimes = np.array(runtimes)
rtf_list = np.array(rtf_list)

asr_rtf = asr_time / total_audio_duration
avg_latency = runtimes.mean()
std_latency = runtimes.std()

mean_rtf = rtf_list.mean()
std_rtf = rtf_list.std()

wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

speedup_vs_realtime = 1 / asr_rtf

# -----------------------------
# PRINT SUMMARY
# -----------------------------
print("\n" + "="*50)
print("FULL ASR EVALUATION SUMMARY")
print("="*50)
print(f"Samples: {len(predictions)}")
print(f"WER: {wer_score:.4f}")
print(f"CER: {cer_score:.4f}")
print(f"ASR RTF (aggregate): {asr_rtf:.4f}")
print(f"Mean RTF: {mean_rtf:.4f}")
print(f"Std RTF: {std_rtf:.4f}")
print(f"Avg latency: {avg_latency:.4f} sec")
print(f"Std latency: {std_latency:.4f}")
print(f"Speedup vs realtime: {speedup_vs_realtime:.2f}x")
print("="*50)

# -----------------------------
# SAVE RESULTS (PER SAMPLE)
# -----------------------------
results_df = pd.DataFrame({
    "audio_path": audio_paths,
    "reference": references,
    "prediction": predictions,
    "sample_cer": sample_cers
})

SAVE_PATH = "/content/drive/MyDrive/tts_asr_results_full.csv"

results_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("Saved per-sample results to:")
print(SAVE_PATH)

# -----------------------------
# SAVE SUMMARY
# -----------------------------
summary_df = pd.DataFrame([{
    "model": "Piper_Amir_Medium",

    "samples": len(predictions),

    "wer": wer_score,
    "cer": cer_score,

    "asr_rtf": asr_rtf,
    "mean_rtf": mean_rtf,
    "std_rtf": std_rtf,

    "avg_latency_sec": avg_latency,
    "std_latency_sec": std_latency,

    "speedup_vs_realtime": speedup_vs_realtime
}])

SUMMARY_PATH = "/content/drive/MyDrive/tts_asr_summary_piper.csv"

summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Saved summary to:")
print(SUMMARY_PATH)

Starting FULL ASR Evaluation...


  0%|          | 0/1052 [00:00<?, ?it/s][NeMo W 2026-06-07 08:57:37 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-07 08:57:37 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 1it [00:00, 14.30it/s]
  0%|          | 1/1052 [00:00<02:00,  8.74it/s][NeMo W 2026-06-07 08:57:37 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-07 08:57:37 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenizat


FULL ASR EVALUATION SUMMARY
Samples: 1052
WER: 0.1071
CER: 0.0455
ASR RTF (aggregate): 0.0593
Mean RTF: 0.0716
Std RTF: 0.0409
Avg latency: 0.1272 sec
Std latency: 0.0316
Speedup vs realtime: 16.87x
Saved per-sample results to:
/content/drive/MyDrive/tts_asr_results_full.csv
Saved summary to:
/content/drive/MyDrive/tts_asr_summary_piper.csv
